# Convert a source to shards

Run on a CPU runtime. Set `SOURCE`, `REPO_SHA` and the archive URLs (see `experiments/DATA_SOURCES.md`) in the next cell, then run the cells in order.

Conversion never reads through the Drive mount: the archives are downloaded (or copied once from Drive) to `/content`, extracted and converted on local disk, and only the finished shards are written back to Drive.

The repo is private: add a read-only GitHub token (Contents: read) under Colab Secrets (the key icon) as `GITHUB_TOKEN`, with notebook access on.

In [ ]:
SOURCE = "neon"
REPO_SHA = ""
ZENODO = "https://zenodo.org/api/records/5914554/files"  # NeonTreeEvaluation v0.2.2
ARCHIVES = [
    {"name": "annotations.zip", "url": f"{ZENODO}/annotations.zip/content"},
    {"name": "evaluation.zip", "url": f"{ZENODO}/evaluation.zip/content"},
    {"name": "training.zip", "url": f"{ZENODO}/training.zip/content"},
]
RESOLVE_ZIP = "raw/resolve/Ecoregions2017.zip"  # relative to the Drive root
VAL_RATIO = 0.15

In [ ]:
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/fair_trees")

assert REPO_SHA, "set REPO_SHA to the pushed commit"
# the repo is private: a read-only GitHub token in Colab Secrets (key icon), named GITHUB_TOKEN
REPO = "github.com/OmdenaAI/HOTOSM-fAIr-Trees.git"
token = userdata.get("GITHUB_TOKEN")
!rm -rf /content/fair_trees
# git clone through subprocess so the token never reaches the cell output
url = f"https://x-access-token:{token}@{REPO}"
subprocess.run(["git", "clone", "-q", url, "/content/fair_trees"], check=True)
# drop the token from the saved remote so it is not left in .git/config
git = ["git", "-C", "/content/fair_trees"]
subprocess.run([*git, "remote", "set-url", "origin", f"https://{REPO}"], check=True)
subprocess.run([*git, "checkout", "-q", REPO_SHA], check=True)
del token, url

%pip install /content/fair_trees -r /content/fair_trees/experiments/requirements.txt
sys.path.insert(0, "/content/fair_trees")

In [ ]:
import hashlib
import json
import shutil
import urllib.request


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


raw_drive = DRIVE / "raw" / SOURCE
manifest_path = raw_drive / "manifest.json"
local_archives = []
if manifest_path.exists():
    for entry in json.loads(manifest_path.read_text())["files"]:
        archive = Path("/content") / entry["name"]
        shutil.copyfile(raw_drive / entry["name"], archive)
        assert sha256_of(archive) == entry["sha256"], f"{entry['name']} differs from the sha256 on Drive"
        local_archives.append(archive)
else:
    raw_drive.mkdir(parents=True, exist_ok=True)
    files = []
    for item in ARCHIVES:
        archive = Path("/content") / item["name"]
        urllib.request.urlretrieve(item["url"], archive)
        shutil.copyfile(archive, raw_drive / item["name"])
        files.append({"name": item["name"], "sha256": sha256_of(archive)})
        local_archives.append(archive)
    manifest_path.write_text(json.dumps({"files": files}, indent=2))
for archive in local_archives:
    print(archive, archive.stat().st_size, "bytes")

In [ ]:
import zipfile

raw_root = Path("/content/raw") / SOURCE
shutil.rmtree(raw_root, ignore_errors=True)  # a rerun extracts into a clean tree
raw_root.mkdir(parents=True)
for archive in local_archives:
    with zipfile.ZipFile(archive) as z:
        members = [m for m in z.namelist() if not m.startswith("__MACOSX/")]
        # annotations.zip and evaluation.zip carry their own top folder; training.zip has RGB/, CHM/, ...
        # at its root, so it goes under training/
        nested = all(m.startswith(f"{archive.stem}/") for m in members)
        z.extractall(raw_root if nested else raw_root / archive.stem, members=members)
for part in ("training/RGB", "annotations", "evaluation/RGB"):
    assert (raw_root / part).is_dir(), f"{raw_root / part} is missing after extraction"

resolve_dir = Path("/content/resolve")
resolve_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(DRIVE / RESOLVE_ZIP) as z:
    z.extractall(resolve_dir)
print(sorted(p.name for p in resolve_dir.rglob("*.shp")))

In [ ]:
from experiments.biome import resolve_biome_fn
from experiments.converters.neon import convert

shapefile = next(resolve_dir.rglob("*.shp"))
converted = Path("/content/converted") / SOURCE
# scratch output on the Colab VM only; convert refuses a non-empty output directory
shutil.rmtree(converted, ignore_errors=True)
shutil.rmtree(converted.parent / f"{converted.name}-work", ignore_errors=True)
stats = convert(
    raw_root,
    converted,
    val_ratio=VAL_RATIO,
    biome_fn=resolve_biome_fn(shapefile),
)
print(stats)

In [ ]:
from experiments.shards import manifest_hash, write_shards

shard_dir = DRIVE / "shards" / SOURCE
write_shards(converted, shard_dir, source=SOURCE, converter_sha=REPO_SHA)
print("manifest sha256:", manifest_hash(shard_dir / "manifest.json"))
shutil.copyfile(converted / "stats.json", shard_dir / "stats.json")

In [ ]:
import random

from affine import Affine
from IPython.display import Image, display
from PIL import Image as PILImage
from PIL import ImageDraw

from fair_trees.dataset import labels_to_boxes, load_geojson
from fair_trees.georef import tile_bounds_3857
from fair_trees.preprocess import read_chip

THUMB = 256
chip_paths = sorted(converted.glob("*/z*/chips/*.tif"))
picked = random.Random(0).sample(chip_paths, min(16, len(chip_paths)))
contact = PILImage.new("RGB", (4 * THUMB, 4 * THUMB))
for i, path in enumerate(picked):
    rgb, _, (x, y, z) = read_chip(path, use_sidecar=False)
    minx, miny, maxx, maxy = tile_bounds_3857(x, y, z)
    res = (maxx - minx) / rgb.shape[1]
    transform = Affine(res, 0, minx, 0, -res, maxy)
    boxes = labels_to_boxes(load_geojson(path.parents[2] / "labels.geojson"), transform)
    tile = PILImage.fromarray(rgb)
    draw = ImageDraw.Draw(tile)
    for x0, y0, x1, y1 in boxes:
        draw.rectangle((x0, y0, x1, y1), outline=(255, 0, 0), width=2)
    contact.paste(tile.resize((THUMB, THUMB)), ((i % 4) * THUMB, (i // 4) * THUMB))
sheet = shard_dir / "qa-contact-sheet.png"
contact.save(sheet)
display(Image(filename=str(sheet)))

In [ ]:
import time

largest = max((p for p in shard_dir.iterdir() if p.suffix == ".tar"), key=lambda p: p.stat().st_size)
start = time.monotonic()
shutil.copyfile(largest, Path("/content") / largest.name)
elapsed = time.monotonic() - start
print(f"{largest.name}: {largest.stat().st_size / 1e6 / elapsed:.1f} MB/s")
# Record this in experiments/DATA_SOURCES.md; restaging must take under 10 minutes.

In [ ]:
drive.flush_and_unmount()